In [1]:
from useful_functions import format_datetime64, get_daterange_str

import pandas as pd
import xarray as xr
from datetime import datetime

from metpy.calc import wind_direction, wind_speed
from metpy.units import units
from ICONProcessor import ICONDataGrid

data_out_dir = '../data_out/'

In [2]:
def get_var_attrs_ref(ref_ds_file, ref_var, res):
    # get reference dataset for meta data
    ref_ds = xr.open_dataset(ref_ds_file)
    attrs = {
        "long_name": ref_ds[ref_var].long_name,
        "standard_name": ref_ds[ref_var].standard_name,
        "units": ref_ds[ref_var].units,
        "temporal_resolution": f'{res} averages (origin is the last value of the timeseries)'
    }
    return attrs

## Script settings

In [3]:
# ICON
icon_nlev = 75 # lowest number of ICON model levels
icon_cid = 35704
icon_exp = 'v370_2030_v2'
icon_file_extpar = '/Users/geoalxx/Python/_ICON_data/hef/grid/external_parameter_icon_hef_51m_DOM07_ALBEDO_RGI_2030_tiles.nc'
#icon_file_30min = f'/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/{icon_exp}_LES_30min_51m_ml_{icon_cid}.nc'  # data
#icon_file = f'/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/{icon_exp}_LES_51m_ml_{icon_cid}.nc'  # z_mc, z_ifc
icon_file_30min = f'/Users/geoalxx/Python/glacier_space/data_icon_h3/nesh_{icon_exp}/{icon_exp}_LES_30min_51m_ml_{icon_cid}.nc'  # data

# WindRanger
wr_file = '/Users/geoalxx/Python/glacier_space/data_obs_h3/windranger/Windranger_cleaned_30min.nc'
wr_file_ref = '/Users/geoalxx/HEFEX III/WindRanger/20250807/20250807_000000.nc'  # used for meta data reference (variable attributes)

## Load and prepare WindRanger data

In [4]:
wr_variables_ts = ['alt_instrument','lat','lon', 'heading', 'pitch', 'roll']
wr_variables_2D = ['dir','MSE','QUALITY','SNR','spd','SPW','u','v','w','VALID']

ds = xr.open_dataset(wr_file)

# collect dimension values
dim_h_wr = ds.height.values
dim_time_wr = pd.to_datetime(ds.time.values, utc=True)

# collect time series data
wr_df_ts = pd.DataFrame(index=dim_time_wr)
for v in wr_variables_ts:
    wr_df_ts[v] = ds[v].values[:, 0]

# collect 2D data
wr_df_2D = {}
for v in wr_variables_2D:
    wr_df_2D[v] = pd.DataFrame(ds[v].values, index=dim_time_wr, columns=dim_h_wr)

## Load and prepare ICON data

In [6]:
ds_30min = xr.open_dataset(icon_file_30min)
#ds = xr.open_dataset(icon_file)
ds_extpar = xr.open_dataset(icon_file_extpar)
ds_cell = ds_extpar.isel(cell=icon_cid-1)

# dimension: time, height (full level and half level)
dim_time_i = pd.Series([ICONDataGrid.convert_float_dt(f) for f in ds_30min.time.values])
elev = ds_30min.z_ifc.values[-1, 0]
dim_h_mc = ds_30min.z_mc.values[-icon_nlev:, 0] - elev
dim_h_ifc = ds_30min.z_ifc.values[-icon_nlev-1:, 0] - elev
print(f'Vertical extent ({icon_nlev} levels):',dim_h_ifc[-1],'-',dim_h_ifc[0],'m')

# get data
icon_df_u = pd.DataFrame(ds_30min.u.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_v = pd.DataFrame(ds_30min.v.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_w = pd.DataFrame(ds_30min.w.values[:, -icon_nlev - 1:, 0], index=dim_time_i, columns=dim_h_ifc)

# calc wind direction and speed
icon_df_wspd = pd.DataFrame(wind_speed(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)
icon_df_wdir = pd.DataFrame(wind_direction(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)

Vertical extent (75 levels): 0.0 - 2611.4626 m


 ## Build datasets

In [7]:
def create_data_array_2D(df, name, time_dim, height_dim, height_name, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in df.index.to_numpy()]

    # create DataArray
    xr_data = xr.DataArray(
        df.values,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            height_dim: (height_dim, df.columns.to_numpy(), {"long_name": height_name}),
        },
        dims=[time_dim, height_dim],
        name=name,
        attrs=var_attrs
    )
    return xr_data

def create_data_array_ts(s, name, time_dim, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in s.index.to_numpy()]

    # create DataArray
    xr_data = xr.DataArray(
        s.values,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
        },
        dims=[time_dim],
        name=name,
        attrs=var_attrs
    )
    return xr_data

### ICON

In [8]:
# Combine DataArrays
res = '30min'
ds_out = xr.Dataset({
    'u'    : create_data_array_2D(icon_df_u, 'u', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(icon_file_30min, 'u', res)),
    'v'    : create_data_array_2D(icon_df_v, 'v', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(icon_file_30min, 'v', res)),
    'w'    : create_data_array_2D(icon_df_w, 'w', 'time', 'height_ifc', 'height at half level center', get_var_attrs_ref(icon_file_30min, 'w', res)),
    'wspd' : create_data_array_2D(icon_df_wspd, 'wspd', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(wr_file_ref, 'VEL', res)),
    'wdir' : create_data_array_2D(icon_df_wdir, 'wdir', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(wr_file_ref, 'DIR', res)),
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="30min")
ds_out = ds_out.reindex(time=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Vertical wind profile from ICON-LES for HEFEX III campaign at Lidar location",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "experiment_id": f"hefex3/{icon_exp}/exp_R3B15_51m",
    "grid_cell": f"cell {icon_cid}/{ds_extpar.sizes['cell']} of domain hef_51m_DOM07.nc (lon, lat, elev): {ds_cell.lon.values:.6f}°E, {ds_cell.lat.values:.6f}°N, {ds_cell.topography_c.values:.2f}m",
    "StartTime": format_datetime64( ds_out.time.values[0] ),
    "StopTime":  format_datetime64( ds_out.time.values[-1] ),
    "comment": f"ICON simulation based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'icon/HEFEX3__ICON_{icon_exp}_cid{icon_cid}_{icon_nlev}ml__avg{res}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/icon/HEFEX3__ICON_v370_2030_v2_cid35704_75ml__avg30min_20250805-20250811.nc.


In [9]:
ds_out

<xarray.Dataset> Size: 481kB
Dimensions:     (time: 318, height_mc: 75, height_ifc: 76)
Coordinates:
  * time        (time) datetime64[ns] 3kB 2025-08-05T00:30:00 ... 2025-08-11T...
  * height_mc   (height_mc) float32 300B 2.575e+03 2.503e+03 ... 15.0 5.0
  * height_ifc  (height_ifc) float32 304B 2.611e+03 2.539e+03 ... 10.0 0.0
Data variables:
    u           (time, height_mc) float32 95kB 2.537 2.674 ... 0.3997 0.8169
    v           (time, height_mc) float32 95kB -18.73 -18.85 ... 2.058 2.61
    w           (time, height_ifc) float32 97kB 0.384 0.3977 ... -0.05945
    wspd        (time, height_mc) float32 95kB 18.9 19.04 19.12 ... 2.097 2.735
    wdir        (time, height_mc) float32 95kB 352.3 351.9 351.7 ... 191.0 197.4
Attributes:
    title:          Vertical wind profile from ICON-LES for HEFEX III campaig...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-11-10
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    campaign:       HEFEX III
    experiment_id:  hefex3/v370_2030_v2/exp_R3B15_51m
    grid_cell:      cell 35704/88640 of domain hef_51m_DOM07.nc (lon, lat, el...
    StartTime:      2025-08-05 00:30:00
    StopTime:       2025-08-11 15:00:00
    comment:        ICON simulation based on glacier outlines from OGGM SSP37...

### Observations from WindRanger

In [10]:
# empty dataset
res = '30min'
ds_out = xr.Dataset()

# add WindRanger data 2D
for v in wr_variables_2D:
    if v == 'dir':
        var_name = 'wdir'
        ref_name = 'DIR'
    elif v == 'spd':
        var_name = 'wspd'
        ref_name = 'VEL'
    else:
        var_name = str.lower(v)
        ref_name = str.upper(v)
    ds_out[var_name] = create_data_array_2D(wr_df_2D[v], var_name, 'time', 'height', 'WindRanger measurement height', get_var_attrs_ref(wr_file_ref, ref_name, res))

# add WindRanger time series data
for v in ['heading','pitch','roll']:
    var_name = str.lower(v)
    ds_out[var_name] = create_data_array_ts(wr_df_ts[v], var_name, 'time', get_var_attrs_ref(wr_file_ref, v, res))

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq=res)
ds_out = ds_out.reindex(time=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Vertical wind profiles from WindRanger lidar acquired during HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"METEK LiDAR Averaged Profile Data ({res}-averaged output from device)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761), leopold.schlagbauer@student.uibk.ac.at (ORCID: 0009-0001-4394-2781)",
    "campaign": "HEFEX III",
    "location": f"WindRanger (lon, lat, elev): {wr_df_ts['lon'].mean():.6f}°E, {wr_df_ts['lat'].mean():.6f}°N, {wr_df_ts['alt_instrument'].mean():.2f}m",
    "StartTime": format_datetime64( ds_out.time.values[0] ),
    "StopTime":  format_datetime64( ds_out.time.values[-1] ),
    "comment": f"Raw WindRanger output for {res} averages (no filtering or correction); Instantaneous values available on request"
}

# Write
output_file = data_out_dir + f'obs/HEFEX3__Obs_WindRanger_L1__avg{res}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/obs/HEFEX3__Obs_WindRanger_L1__avg30min_20250807-20250831.nc.


In [11]:
ds_out

<xarray.Dataset> Size: 876kB
Dimensions:  (time: 1171, height: 13)
Coordinates:
  * time     (time) datetime64[ns] 9kB 2025-08-07T00:30:00 ... 2025-08-31T09:...
  * height   (height) float32 52B 7.0 10.0 15.0 20.0 ... 125.0 150.0 175.0 200.0
Data variables: (12/13)
    wdir     (time, height) float32 61kB 228.0 229.6 229.5 229.8 ... nan nan nan
    mse      (time, height) float64 122kB 0.01643 0.01596 0.01418 ... nan nan
    quality  (time, height) float32 61kB 0.0 0.0 0.0 0.0 0.0 ... nan nan nan nan
    snr      (time, height) float64 122kB -17.21 -16.57 -16.32 ... nan nan nan
    wspd     (time, height) float32 61kB 4.402 4.339 4.686 4.623 ... nan nan nan
    spw      (time, height) float64 122kB 2.004 2.372 3.024 ... nan nan nan
    ...       ...
    v        (time, height) float32 61kB 2.944 2.813 3.046 2.985 ... nan nan nan
    w        (time, height) float32 61kB -0.3755 -0.4162 -0.4263 ... nan nan nan
    valid    (time, height) float64 122kB 0.9848 0.9954 0.9991 ... nan nan nan
    heading  (time) float32 5kB 0.0 0.0 0.0 0.0 0.0 0.0 ... nan nan nan nan nan
    pitch    (time) float32 5kB 0.03929 0.03597 0.03434 0.03472 ... nan nan nan
    roll     (time) float32 5kB -1.064 -1.074 -1.095 -1.101 ... nan nan nan nan
Attributes:
    title:        Vertical wind profiles from WindRanger lidar acquired durin...
    institution:  Humboldt-Universität zu Berlin
    source:       METEK LiDAR Averaged Profile Data (30min-averaged output fr...
    history:      Created 2026-11-10
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX III
    location:     WindRanger (lon, lat, elev): 10.771262°E, 46.799877°N, 2720...
    StartTime:    2025-08-07 00:30:00
    StopTime:     2025-08-31 09:30:00
    comment:      Raw WindRanger output for 30min averages (no filtering or c...

In [12]:
ds_out.close()